# Build Constructors Dimension

1. Read silver `constructors` table
1. Read gold `ref_nationality_region` table
1. Join the data from `constructors` with `ref_nationality_region` using `nationality`
1. Select the required columns
    - constructors.constructor_id
    - constructors.constructor_name
    - constructors.nationality
    - ref_nationality_region.region
1. Write the transformed data to gold `dim_constructors` table



#### Entity Relationship Diagram - Formula1 Silver Schema

![Formula1 Silver Data.png](../../z-course-images/formula1-silver-data-erd.png "Formula1 Silver Data.png")


#### Entity Relationship Diagram - Formula1 Gold Schema

![Formula1 Gold Data.png](../../z-course-images/formula1-gold-data-erd.png "Formula1 Gold Data.png")

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")  

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read source tables
- `silver.constructors`
- `gold.ref_nationality_region`

In [0]:
constructors_df = (spark.table(f"{catalog_name}.{silver_schema}.constructors")
                   .filter(F.col("batch_id") == v_batch_id)
                   )

In [0]:

ref_nationality_region_df = (spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")
                             
                             )

#### Step 2 - Join `constructors` with `nationality_region_df` using `nationality`
Select the following columns   
1. constructors.constructor_id 
1. constructors.constructor_name 
1. constructors.nationality 
1. ref_nationality_region.region

In [0]:
dim_constructors_df = (
    constructors_df
        .join(
            ref_nationality_region_df,
            constructors_df.nationality == ref_nationality_region_df.nationality,
            "left"
            )
        .select (
            constructors_df.constructor_id,
            constructors_df.constructor_name,
            constructors_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)

In [0]:
display(dim_constructors_df)

constructor_id,constructor_name,nationality,nationality_region
ats,ATS,Italian,Europe
benetton,Benetton,Italian,Europe
bmw,BMW,German,Europe
brabham-repco,Brabham-Repco,British,Europe
cadillac,Cadillac F1 Team,American,North America
force_india,Force India,Indian,Asia
lotus-pw,Lotus-Pratt & Whitney,British,Europe
osella,Osella,Italian,Europe
token,Token,British,Europe
amon,Amon,New Zealander,Oceania


#### Step 3 - Write the transformed data to the `gold` `dim_constructors` table

In [0]:
write_to_gold(
    input_df=dim_constructors_df,
    target_table=target_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
display(spark.table(target_table))

constructor_id,constructor_name,nationality,nationality_region,created_timestamp,updated_timestamp
ats,ATS,Italian,Europe,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
benetton,Benetton,Italian,Europe,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
bmw,BMW,German,Europe,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
brabham-repco,Brabham-Repco,British,Europe,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
cadillac,Cadillac F1 Team,American,North America,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
force_india,Force India,Indian,Asia,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
lotus-pw,Lotus-Pratt & Whitney,British,Europe,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
osella,Osella,Italian,Europe,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
token,Token,British,Europe,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
amon,Amon,New Zealander,Oceania,2026-11-08T17:03:35.751Z,2026-11-08T17:03:35.751Z
